<a href="https://colab.research.google.com/github/Alfred9/Exploring-LLMs/blob/main/AI_Agents/Langchain_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -qU langchain_community
!pip install -qU duckduckgo-search
!pip install -U langchain langchain-community ddgs

In [9]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke("Who is the current UFC Heavyweight champion 2026?")
print(results)

List ofcurrentUFCfighters This list ofcurrentUFCfighters recordscurrentUltimate Fighting Championship (UFC) fighters' information, country origins, recent fighter signings and departures, fight schedules and results and thechampionof each division. As of 28 February2026,theUFCroster consisted of fighters from 76 countries. [1] TheUltimate Fighting Championship (UFC) recognizes 12 active divisions — eight for men and four for women — each with its own reigning titleholder. This regularly updated list ofcurrentUFCchampions(2026) presents every division'schampion, along with dates won and title defense records. Explore the full division-wise breakdown of allUFCtitle holders in one place. UFCChampionsList2026: All 12currenttitle holders by division, all-time title defense records, and emergingchampions. Tom Aspinall, Ilia Topuria, and more. That night, Chuck Liddell lost the lightheavyweightbelt to Quinton "Rampage" Jackson, leaving theUFCwith zerochampionswith title defenses for over a mo

In [10]:
from langchain_core.tools import Tool
import random

def get_weather_info(location: str) -> str:
  """Fetches dummy waether informatiion for a given location"""

  weather_conditions = [
     {"condition": "Rainy", "temp_c": 15},
     {"condition": "Clear", "temp_c": 25},
     {"condition": "Windy", "temp_c": 20}
  ]

  data = random.choice(weather_conditions)
  return f"Weather in {location}: {data['condition']}, {data['temp_c']}°C"


  #initialize the tool
  weather_info_tool = Tool (
      name="get_weather_info",
      func=get_weather_info,
      description="Fetches dummy weather information for a given location."
  )


In [11]:
from langchain_core.tools import Tool
from huggingface_hub import list_models

def get_hub_stats(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        # List models from the specified author, sorted by downloads
        models = list(list_models(author=author, sort="downloads", direction=-1, limit=1))

        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        else:
            return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

# Initialize the tool
hub_stats_tool = Tool(
    name="get_hub_stats",
    func=get_hub_stats,
    description="Fetches the most downloaded model from a specific author on the Hugging Face Hub."
)

# Example usage
print(hub_stats_tool.invoke("facebook")) # Example: Get the most downloaded model by Facebook

Error fetching models for facebook: HfApi.list_models() got an unexpected keyword argument 'direction'


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Generate the chat interface, including the tools
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
)

chat = ChatHuggingFace(llm=llm, verbose=True)
tools = [search_tool, weather_info_tool, hub_stats_tool]
chat_with_tools = chat.bind_tools(tools)

# Generate the AgentState and Agent graph
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])],
    }

## The graph
builder = StateGraph(AgentState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message requires a tool, route to tools
    # Otherwise, provide a direct response
    tools_condition,
)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

messages = [HumanMessage(content="Who is Facebook and what's their most popular model?")]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)